<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Retrieval Augmented Generation
</b></font> </br></p>


---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()
# LangSmith Tracing
run_cfg = {
    "run_name": "M08_RAG_LangChain",
    "tags": ["m08", "rag"],
    "metadata": {"notebook": "M08", "version": "1.0"}
}


In [2]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
    ("markitdown[all]", "markitdown"),
    "langchain_huggingface",
    ("langchain-text-splitters", "langchain_text_splitters"),
])


🔄 Installiere markitdown[all]...
✅ markitdown[all] erfolgreich installiert und importiert
🔄 Installiere langchain_huggingface...
✅ langchain_huggingface erfolgreich installiert und importiert
✅ langchain_text_splitters bereits verfügbar


In [3]:
#@title 📂 Dokumente { display-mode: "form" }
import shutil
from genai_lib.utilities import copy_from_github

shutil.rmtree("files", ignore_errors=True)
copy_from_github(
    source="ralf-42/GenAI/02_daten/01_text",
    target="files",
    mask="biografien_*",
)

  Kopiere: 02_daten/01_text/biografien_1.txt → files/biografien_1.txt
  Kopiere: 02_daten/01_text/biografien_2.md → files/biografien_2.md
  Kopiere: 02_daten/01_text/biografien_3.pdf → files/biografien_3.pdf
  Kopiere: 02_daten/01_text/biografien_4.docx → files/biografien_4.docx

Ergebnis: 4 Datei(en) kopiert → files


['files/biografien_1.txt',
 'files/biografien_2.md',
 'files/biografien_3.pdf',
 'files/biografien_4.docx']

# 1 | Übersicht
---

In [ ]:
from genai_lib.utilities import mermaid

# RAG-Architektur
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    DOC([Dokumente]) --> SPLIT[Text Splitter]
    SPLIT --> EMBED[Embeddings]
    EMBED --> DB[(Vektordatenbank)]
    QUERY([Nutzeranfrage]) --> QEMBED[Embedding]
    QEMBED --> RETRIEVE[Retriever]
    DB --> RETRIEVE
    RETRIEVE --> PROMPT[Prompt + Kontext]
    QUERY --> PROMPT
    PROMPT --> LLM[LLM]
    LLM --> ANT([Antwort])
    style DOC fill:#FF9800,color:#000
    style ANT fill:#4CAF50,color:#000
"""
mermaid(diagram, width=700)


<p><font color='black' size="5">
Einführung in Retrieval-Augmented Generation (RAG)
</font></p>

RAG (Retrieval-Augmented Generation) ergänzt ein LLM gezielt mit externem Wissen zur Laufzeit — ohne das Modell neu zu trainieren. Das unterscheidet RAG grundlegend von Fine-Tuning: Während Fine-Tuning Verhalten und Stil des Modells dauerhaft verändert, bleibt das Modell bei RAG unberührt — die Wissensbasis wird stattdessen über den Kontext eingespeist.

In der Praxis bedeutet das: Dokumente, Datenbanken oder interne Handbücher, die ein Modell nie gesehen hat, können trotzdem als Grundlage für präzise Antworten dienen. Entscheidend ist dabei die Qualität des Retrieval-Schritts — welche Dokumente gefunden werden, bestimmt direkt, was das Modell antworten kann und was nicht.

**Praktischer Anwendungsfall: Kundenanfrage**

[Bild Kundenanfrage](https://www.researchgate.net/publication/378467192/figure/fig3/AS:11431281235857152@1712887907223/Zusammenspiel-zwischen-Vektordatenbank-Kundenanfrage-und-LLM.png)


Quelle: [Die_Nutzung_von_ChatGPT_in_Unternehmen](https://www.researchgate.net/publication/378467192_Die_Nutzung_von_ChatGPT_in_Unternehmen_Ein_Fallbeispiel_zur_Neugestaltung_von_Serviceprozessen)

# 2 | RAG-Prozess
---

Der RAG-Prozess gliedert sich in zwei Teilprozesse: **Datensammlung** zur Vorbereitung externer Wissensquellen und **Abruf & Erweiterung** zur kontextgestützten Beantwortung von Nutzeranfragen. Beide Schritte müssen sauber ineinandergreifen — Schwächen in der Datensammlung (z. B. schlechtes Chunking) wirken sich direkt auf die Antwortqualität aus.

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/rag_process.png" width="600" alt="Avatar">

<p><font color='black' size="5">
Datensammlung
</font></p>

Relevante Texte — Fachartikel, Handbücher, Webseiten — werden zunächst gesammelt und dann in kleinere, zusammenhängende Abschnitte (*Chunks*) aufgeteilt. Ein Embedding-Modell wandelt jeden Chunk in einen numerischen Vektor um. Diese Vektoren werden zusammen mit dem ursprünglichen Text in einer Vektordatenbank (z. B. FAISS oder ChromaDB) gespeichert, damit sie später effizient durchsucht werden können.

<p><font color='black' size="5">
Abruf & Erweiterung
</font></p>

Eine Nutzeranfrage wird zunächst in ein Embedding konvertiert. Ein Retriever durchsucht damit die Vektordatenbank und liefert die inhaltlich ähnlichsten Dokumente zurück. Diese Dokumente ergänzen die ursprüngliche Anfrage im Prompt — das Sprachmodell erhält also nicht nur die Frage, sondern auch den passenden Kontext. Auf dieser Grundlage generiert es eine fundierte Antwort.

Häufig wird unterschätzt, wie stark die Retrieval-Qualität das Endergebnis prägt: Findet der Retriever die falschen Chunks — etwa wegen zu allgemeiner Queries, schlechtem Chunking oder einem unpassenden Embedding-Modell — kann auch ein gutes LLM keine korrekte Antwort liefern.


<p><font color='black' size="5">
Einschränkungen von RAG
</font></p>

RAG entfaltet seinen Nutzen dort, wo das Basismodell keine ausreichenden Informationen hat: bei domänenspezifischen, proprietären oder sehr aktuellen Daten aus Bereichen wie Finanzen, Recht oder interner Unternehmenskommunikation. Für allgemeines Wissen, das bereits im Vortraining abgedeckt ist, bringt RAG keinen Mehrwert — der unnötige Retrieve kostet Rechenzeit und Latenz, ohne die Antwortqualität zu verbessern.

RAG scheitert außerdem, wenn die Retrieval-Pipeline selbst schwach ist: zu grobe Chunks, die keinen kohärenten Kontext liefern; ein Embedding-Modell, das die Domänensprache nicht versteht; oder zu allgemeine Suchanfragen, die semantisch nicht trennscharf genug sind. In solchen Fällen sollte zunächst die Datenpipeline verbessert werden, bevor das LLM optimiert wird.

[RAG-Visualizer](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)

# 3 | Deep Dive: Token & Chunks
---

<p><font color='black' size="5">
Tokenizing
</font></p>

**Tokenisierung und Chunking — Grundlage jedes RAG-Systems**

LLMs verarbeiten Text nicht als Sätze oder Absätze, sondern als Folgen von *Tokens*. Tokenisierung und Chunking sind daher keine optionalen Vorverarbeitungsschritte — sie bestimmen, welche Informationen das Modell überhaupt sehen kann.

**Tokenisierung: Die Grundlage der Textverarbeitung**

Tokenisierung zerlegt Text in kleinere Einheiten — sogenannte Tokens — die Wörter, Teilwörter oder einzelne Zeichen sein können. Da LLMs eine begrenzte Eingabelänge haben, sorgt die Tokenisierung dafür, dass Texte in einem einheitlichen Format vorliegen. Tokens mit ähnlicher Bedeutung oder Verwendung erhalten ähnliche Repräsentationen, was dem Modell hilft, semantische Zusammenhänge zu erkennen.

**Richtwert:** 1 DIN-A4-Seite entspricht ca. 300 Wörtern und ca. 450 Token.

[OpenAI Tokenizer](https://platform.openai.com/tokenizer)

<p><font color='black' size="5">
Chunking
</font></p>

**Chunking: Texte in verdauliche Häppchen teilen**

Chunking ist der Prozess, bei dem längere Texte in kleinere, zusammenhängende Abschnitte (Chunks) aufgeteilt werden.

**Warum ist Chunking wichtig?**

1. **Verarbeitung langer Texte:** Da LLMs eine begrenzte Eingabelänge haben, ermöglicht Chunking die Verarbeitung von Texten, die länger als diese Grenze sind.

2. **Kontexterhaltung:** Durch geschicktes Chunking kann der relevante Kontext innerhalb eines Chunks erhalten bleiben, was für viele NLP-Aufgaben entscheidend ist.

3. **Effizienz:** Kleinere Chunks können parallel verarbeitet werden, was die Gesamtverarbeitungszeit reduzieren kann.



**Tokenisierungs- und Chunking-Strategien im Überblick**

Die gebräuchlichsten Tokenisierungsverfahren sind **wortbasiert** (einfach, aber bei komplexen Sprachen oft unzureichend), **Subword-basiert** (z. B. BPE oder WordPiece — Standard bei modernen LLMs, robust gegenüber unbekannten Wörtern) und **zeichenbasiert** (für Sondersprachen oder sehr niedrige Ressourcen).

Beim Chunking stehen drei Ansätze im Vordergrund: **satzbasiertes Chunking** ist einfach umzusetzen, verliert aber Kontext über Satzgrenzen hinweg. **Überlappende Chunks** mildern dieses Problem durch wiederholten Kontext an den Grenzen. **Semantisches Chunking** ist am aufwändigsten, liefert aber die inhaltlich kohärentesten Einheiten — und damit in der Regel die besten Retrieval-Ergebnisse.

[ChunkViz](https://chunkviz.up.railway.app/)

# 4 | Deep Dive: Embedding
---

Embeddings sind numerische Vektorrepräsentationen von Text (oder anderen Datentypen). Ein Embedding-Modell verarbeitet Eingangsdaten so, dass inhaltlich ähnliche Elemente im Vektorraum nah beieinander liegen — unabhängig davon, ob sie exakt gleiche Wörter verwenden. Das macht Embeddings zur zentralen Grundlage für semantische Suche in RAG-Systemen.



Ein **Embedding** ist ein Vektor aus Gleitkommazahlen, der Ähnlichkeiten zwischen Texten misst. Kleinere Abstände zwischen Vektoren bedeuten eine stärkere inhaltliche Nähe.

[Beispiel Fahrzeug 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)

[Beispiel Fahrzeug 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)



OpenAI bietet zwei Einbettungsmodelle an:
- **text-embedding-3-small**
- **text-embedding-3-large**

**Unterschiede und Auswahlkriterien:**

| Kriterium         | text-embedding-3-small | text-embedding-3-large |
|------------------|----------------------|----------------------|
| **Genauigkeit & Leistung** | Weniger detailliert, aber für viele Aufgaben ausreichend | Erfasst komplexere Zusammenhänge, ideal für anspruchsvolle NLP-Aufgaben |
| **Rechenaufwand** | Effizient, benötigt weniger Ressourcen | Höherer Speicher- und Rechenbedarf |
| **Latenz & Geschwindigkeit** | Schnellere Verarbeitung, geringe Latenz | Höhere Latenz, nicht ideal für Echtzeit-Anwendungen |
| **Kosten** | Kostengünstiger, ideal für skalierbare Anwendungen | Teurer in Betrieb und Bereitstellung |
| **Anwendungsfälle** | Chatbots, Echtzeitsysteme, einfache Textverarbeitung | Semantische Analyse, komplexe NLP-Aufgaben, KI-Forschung |



**Fazit**:  
- **Large**: Wenn hohe Präzision, detailliertes Sprachverständnis und ausreichend Ressourcen vorhanden sind.  
- **Small**: Wenn Effizienz, niedrige Kosten und schnelle Reaktionszeiten im Vordergrund stehen.  

Die Wahl des Modells hängt von den spezifischen Anforderungen der Anwendung ab.


<p><font color='black' size="5">
Instanz eines Einbettungsmodells
</font></p>

Instanz eines Einbettungsmodells mit `text-embedding-3-small`:

In [4]:
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

<p><font color='black' size="5">
Vektoren
</font></p>

Ein Vektor ist eine mathematische Darstellung, die eine Menge von Zahlen in einer bestimmten Reihenfolge speichert. In einem Einbettungsmodell wird jeder Text – sei es ein einzelnes Wort, ein Satz oder ein ganzer Absatz – als Vektor in einem hochdimensionalen Raum dargestellt. Diese Darstellung ermöglicht es, Ähnlichkeiten zwischen Texten mathematisch zu berechnen.

**Wortvektoren**
Wenn ein Einbettungsmodell ein einzelnes Wort in einen Vektor umwandelt, geschieht dies auf Basis der Bedeutung und des Kontexts des Wortes. Das bedeutet, dass Wörter mit ähnlicher Bedeutung oder Verwendung in der Sprache ähnliche Vektoren erhalten. Ein Beispiel:

- Das Wort „Katze“ könnte in einem 3D-Raum (vereinfacht dargestellt) als Vektor **(0.5, 1.2, -0.3)** erscheinen.
- Das Wort „Hund“ könnte einen ähnlichen Vektor haben, z. B. **(0.6, 1.1, -0.2)**, da beide Begriffe semantisch verwandt sind.

Dagegen hätte ein völlig anderes Wort wie „Auto“ einen weit entfernten Vektor, z. B. **(2.3, -0.4, 1.7)**, da es eine ganz andere Bedeutung hat.


In [5]:
vektor1 = embeddings_model.embed_query("Hund")
vektor2 = embeddings_model.embed_query("Ein Hund läuft über die Straße.")

print(type(vektor1))

<class 'list'>


Die Ausgabe ist eine Python-Liste mit Gleitkommazahlen — die Länge entspricht der Vektordimension des gewählten Modells.

In [6]:
print(len(vektor1))
print(len(vektor2))

1536
1536


Die Vektordimension ist bei einem Modell immer konstant — unabhängig von der Länge des Eingabetexts. Das `large`-Modell erzeugt qualitativ bessere Vektoren, nicht notwendigerweise längere. Nachfolgend sind die ersten zehn Elemente dargestellt.

In [7]:
print(vektor1[:5]) # Hund
print(vektor2[:5]) # Ein Hund läuft über die Straße.

[-0.04058837890625, -0.04278564453125, 0.0020923614501953125, 0.01541900634765625, 0.01824951171875]
[-0.0035228729248046875, 0.005580902099609375, -0.0177459716796875, 0.01097869873046875, 0.001613616943359375]


<p><font color='black' size="5">
 Vektoren vergleichen
</font></p>

Zum Vergleich von Vektoren stehen mehrere Methoden zur Verfügung. Die **Kosinus-Ähnlichkeit** misst den Winkel zwischen zwei Vektoren — sie gewichtet die Richtung stärker als die Magnitude und ist die Standardmethode für semantische Textsuche. Die **Euklidische Distanz** gibt die direkte räumliche Entfernung an und kommt häufig in Clustering-Algorithmen zum Einsatz. Die **Manhattan-Distanz** summiert absolute Differenzen und eignet sich vor allem für gitterbasierte Probleme.

OpenAI empfiehlt für Embedding-Vergleiche die Kosinus-Ähnlichkeit, da sie die ursprüngliche Vektorgröße erhält und das Vorzeichen der Komponenten berücksichtigt — beides relevant für die korrekte Ähnlichkeitsinterpretation. Siehe [OpenAI FAQ](https://platform.openai.com/docs/guides/embeddings/frequently-asked-questions).

**Interpretation:**

| Ähnlichkeitsmaß         | Wertebereich | Interpretation                                                                                           |
|-------------------------|--------------|---------------------------------------------------------------------------------------------------------|
| Kosinus-Ähnlichkeit     | -1 bis 1     | - 1: Maximale Ähnlichkeit (identische Richtung) <br> - 0: Keine Ähnlichkeit (orthogonal) <br> - -1: Maximale Unähnlichkeit (entgegengesetzte Richtung) <br> - Höhere Werte bedeuten größere Ähnlichkeit |
| Euklidischer Abstand     | 0 bis ∞      | - 0: Identische Vektoren (maximale Ähnlichkeit) <br> - Je größer der Wert, desto unähnlicher sind die Vektoren <br> - Niedrigere Werte bedeuten größere Ähnlichkeit |
| Manhattan-Distanz       | 0 bis ∞      | - 0: Identische Vektoren (maximale Ähnlichkeit) <br> - Je größer der Wert, desto unähnlicher sind die Vektoren <br> - Niedrigere Werte bedeuten größere Ähnlichkeit |


In [8]:
#@markdown   <p><font size="4" color='green'>  Ähnlichkeit ermitteln</font> </br></p>
# Ermittlung & Ausgabe von Vektoren

import numpy as np
from scipy.spatial.distance import cityblock, cosine, euclidean

def similarity(vektor1, vektor2):
    vector1 = np.array(vektor1)
    vector2 = np.array(vektor2)

    # 1. Kosinus-Ähnlichkeit
    cosine_similarity_value = 1 - cosine(vector1, vector2) # Umbenennung, um Shadowing zu vermeiden
    print(f"Kosinus-Ähnlichkeit: {cosine_similarity_value:.4f}")

    # 2. Euklidischer Abstand
    euclidean_distance_value = euclidean(vector1, vector2) # Umbenennung, um Shadowing zu vermeiden
    print(f"Euklidischer Distanz: {euclidean_distance_value:.4f}")

    # 3. Manhattan-Distanz
    manhattan_distance = cityblock(vector1, vector2)
    print(f"Manhattan Distanz: {manhattan_distance:.4f}")

In [9]:
vektor1 = embeddings_model.embed_query("Ein Hund läuft über eine Brücke")
vektor2 = embeddings_model.embed_query("Ein Hund läuft über die Straße.")
similarity(vektor1, vektor2)

Kosinus-Ähnlichkeit: 0.7660
Euklidischer Distanz: 0.6843
Manhattan Distanz: 21.0725


In [10]:
vektor1 = embeddings_model.embed_query("Ein Hund läuft über eine Brücke.")
vektor2 = embeddings_model.embed_query("Quantenmechanik beschreibt das Verhalten subatomarer Teilchen.")
similarity(vektor1, vektor2)

Kosinus-Ähnlichkeit: 0.1934
Euklidischer Distanz: 1.2701
Manhattan Distanz: 39.1577


[Embedding Projector](https://projector.tensorflow.org/?hl=de)

# 5 | Deep Dive: Vectorstore
---

Ein **Vectorstore** ist eine spezialisierte Datenbank zur Speicherung und schnellen Suche von Texten in Form von Vektoren. Er bildet die Grundlage für semantische Suche in Retrieval-Systemen wie RAG. Durch die Umwandlung von Text in numerische Repräsentationen (Embeddings) können inhaltlich ähnliche Informationen effizient gefunden werden – selbst wenn die exakten Wörter nicht übereinstimmen.


**Beispiel Erstellung & Abfrage eines Vectorstore:**

In [11]:
# Import
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [12]:
# Chroma-Speicherverzeichnis (persistent)
chroma_dir = "chroma_demo"

In [13]:
# OpenAI Embeddings vorbereiten
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [14]:
# 2. Beispiel-Dokumente definieren
texte = [
    "Python ist eine beliebte Programmiersprache für Machine Learning.",
    "Gradio ist ein Python-Framework zur Erstellung von KI-Demos.",
    "ChromaDB ist eine Vektordatenbank zur Ähnlichkeitssuche.",
    "LangChain verbindet Sprachmodelle mit externem Wissen und Tools.",
    "Mit Hugging Face können moderne NLP-Modelle einfach genutzt werden.",
    "OpenAI bietet leistungsstarke APIs für Text-, Bild- und Sprachmodelle.",
    "Vektordatenbanken speichern Embeddings für semantische Suchanfragen.",
    "Retrieval-Augmented Generation kombiniert Suche mit Textgenerierung.",
    "Embeddings wandeln Texte in numerische Repräsentationen um.",
    "ChatGPT kann als intelligenter Assistent in Anwendungen integriert werden."
]

docs = [Document(page_content=t) for t in texte]

In [15]:
# 3. Vektordatenbank mit Chroma erstellen
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=chroma_dir
)

In [16]:
# Eine Beispiel-Query
query = "Was sind Embeddings?"
docs_retrieved_with_scores = vectordb.similarity_search_with_score(query, k=5)   # Die Top 5 Fundstellen werden bereitgestellt

In [17]:
# Ausgabe der gefundenen Texte und Scores
mprint(f"## 🔍 Gefundene Dokumente")
mprint("---")
mprint(f"**Query:** {query}")
print()
mprint(f"**Quellen:**")
for i, (doc, score) in enumerate(docs_retrieved_with_scores, 1):
    print(f"{i}.  Id: {doc.id:40} Inhalt: {doc.page_content} (Score: {score:.4f})")

## 🔍 Gefundene Dokumente

---

**Query:** Was sind Embeddings?

**Quellen:**

1.  Id: 738fe98d-2edf-43b8-9042-f53815a92cab     Inhalt: Embeddings wandeln Texte in numerische Repräsentationen um. (Score: 0.6746)
2.  Id: 3fb3a875-5d3b-4ed5-a118-53bbd0c92672     Inhalt: Vektordatenbanken speichern Embeddings für semantische Suchanfragen. (Score: 0.8059)
3.  Id: efa91ec0-fe61-4aa6-a6ce-2b757cf1ef00     Inhalt: Mit Hugging Face können moderne NLP-Modelle einfach genutzt werden. (Score: 1.2483)
4.  Id: beeb9f01-e981-49ee-ae89-d35be1dd6e5b     Inhalt: Retrieval-Augmented Generation kombiniert Suche mit Textgenerierung. (Score: 1.3182)
5.  Id: ae1d8702-b7ef-4cf2-93af-3bcb458a6da8     Inhalt: LangChain verbindet Sprachmodelle mit externem Wissen und Tools. (Score: 1.3374)


Die Standard-Metrik in Chroma ist die `L2-Distanz` (Euklidische Distanz), genauer gesagt die `quadrierte L2-Distanz ohne Wurzel` aus Effizienzgründen.

# 6 | Hands-On: RAG - Biografien
---

RAG entfaltet seinen größten Nutzen bei Informationen, die das Basismodell nicht kennt — also bei proprietären, internen oder sehr spezifischen Daten. Unternehmen profitieren besonders davon, weil ihre Geschäftsdokumente, Kundendaten und internen Berichte außerhalb des öffentlichen Trainingskorpus liegen.

Als Beispieldatensatz dient ein eigens erstellter Satz synthetischer Biografien von Forscher:innen aus verschiedenen Gebieten. Dieser Datensatz enthält Informationen, die ein Basismodell nicht kennen kann — und eignet sich damit gut, um den Retrieval-Schritt isoliert zu testen und nachzuvollziehen.

In [18]:
# Warnungen unterdrücken, hier nicht relevant
import warnings
warnings.filterwarnings("ignore", message=".*triton.*")

In [19]:
# ── Stdlib ────────────────────────────────────────────────────────────────────
from pathlib import Path
import re

# ── Numerik ───────────────────────────────────────────────────────────────────
import numpy as np
from scipy.spatial.distance import cityblock, cosine, euclidean

# ── Dokumentverarbeitung ──────────────────────────────────────────────────────
from markitdown import MarkItDown

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Pydantic ──────────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field

# ── Modell-Konfiguration ──────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE


<p><font color='black' size="5">
Datensammlung
</font></p>

Die Funktion teilt ein Dokument in Chunks auf und erstellt daraus Embeddings für den RAG-Abruf. `chunk_size` legt die maximale Segmentlänge fest; `chunk_overlap` bestimmt, wie viele Token am Anfang jedes neuen Chunks wiederholt werden — damit bleibt der Kontext an den Chunk-Grenzen erhalten.

Blockgröße 1000 Token, Überlappung 200 Token: Die Überlappung stellt sicher, dass Informationen, die über eine Chunk-Grenze hinausgehen, nicht verloren gehen.

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/rag_process_01.png" width="600" alt="Avatar">


In [20]:
# Dokumente mit MarkItDown laden und als LangChain-Document-Objekte zurückgeben
SUPPORTED_EXTENSIONS = {
    ".txt", ".md", ".pdf", ".docx", ".doc",
    ".html", ".htm", ".csv", ".json", ".xml",
}


def load_documents_from_directory(directory_path: str) -> list[Document]:
    """Lädt unterstützte Dateien und konvertiert sie mit MarkItDown zu Markdown-Text."""
    converter = MarkItDown()
    directory = Path(directory_path)
    documents: list[Document] = []

    for file_path in sorted(directory.rglob("*")):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        try:
            result = converter.convert(str(file_path))
            text = (result.text_content or "").strip()
            if not text:
                print(f"⚠️ Übersprungen (kein Text): {file_path.name}")
                continue

            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": str(file_path)},
                )
            )
            print(f"✓ Geladen: {file_path.name}")
        except Exception as exc:
            print(f"✗ Fehler bei {file_path.name}: {exc}")

    return documents


# Dokumente laden
# In Colab und lokal funktioniert der relative Pfad, weil die Dateien nach "files" kopiert wurden.
directory_path = "files"
documents = load_documents_from_directory(directory_path)


✓ Geladen: biografien_1.txt
✓ Geladen: biografien_2.md
✓ Geladen: biografien_3.pdf
✓ Geladen: biografien_4.docx


In [21]:
type(documents), len(documents)

(list, 4)

Die Biografiedaten werden in ChromaDB geladen. Als Embedding-Modell kommt `text-embedding-3-small` von OpenAI zum Einsatz (`OpenAIEmbeddings`). Die Texte werden mit dem `RecursiveCharacterTextSplitter` in Chunks aufgeteilt und dann eingebettet.

In [22]:
# Text-Splitter konfigurieren und Dokumente aufteilen
chunk_size = 800
chunk_overlap = 80

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents)

In [23]:
type(docs), len(docs)

(list, 69)

In [24]:
# Embeddingsmodell festlegen
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

In [25]:
# Vektordatenbank erstellen und speichern
persistent_directory = "chroma_db"
vectorstore = Chroma.from_documents(docs, embeddings, persist_directory=persistent_directory)


<p><font color='black' size="5">
Abfrage und Erweiterung
</font></p>

Hier wird die **Answer-Chain** mit der **modernen LCEL-Syntax** zusammengebaut.

**Standardansatz (M08):** Eine vollständige RAG-Chain enthält den Retriever als Bestandteil:
```python
chain = (
    {"context": retriever | format_documents, "question": RunnablePassthrough()}
    | rag_prompt | llm | parser
)
```

**Optimierter Ansatz (M08a):** Der Retrieve-Schritt wird aus der Chain herausgezogen, sodass die `answer_chain` nur noch Prompt, LLM und Parser enthält:
```python
answer_chain = rag_prompt | llm | parser
```

**Warum?** Der Retrieve erfolgt separat über `similarity_search_with_score()`, was zwei Vorteile bietet:

1. **Kein doppelter Retrieve:** Die gleichen Dokumente werden zur Anzeige (mit Score) und als LLM-Kontext verwendet
2. **Volle Transparenz:** Man sieht exakt, welche Dokumente das LLM als Kontext erhält

<img src="https://raw.githubusercontent.com/ralf-42/GenAI/main/07_image/rag_process_02.png" width="600" alt="Avatar">


Vordefinierter RAG-Prompt aus dem LangChain Hub — optimiert für Retrieval-Augmented Generation:

[LangChainHub](https://smith.langchain.com/hub/rlm/rag-prompt)

In [26]:
# RAG-Prompt als strukturiertes Template
rag_prompt = load_prompt("https://github.com/ralf-42/GenAI/blob/main/05_prompt/m08_rag_prompt.md")

print("✅ RAG-Prompt Template geladen")

✅ RAG-Prompt Template geladen


**💡 Alternative: LangSmith Prompt Hub (Optional)**

<details>

Statt einen eigenen Prompt zu definieren, lassen sich vordefinierte Prompts aus dem LangSmith Hub laden:

```python
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")
```

**Vorteile:**
- ✅ Community-getestete Prompts
- ✅ Versionierung und A/B-Testing
- ✅ Zentrale Verwaltung

**Nachteil:**
- ⚠️ Erfordert LangSmith-Account und API-Key (funktionierte im Test auch ohne API-Key)

In diesem Notebook wird ein selbst definierter Prompt verwendet — für mehr Transparenz und Unabhängigkeit von externen Diensten.

LLM und Output-Parser werden initialisiert. In dieser optimierten Version gibt es keinen separaten Retriever — der Retrieve-Schritt erfolgt direkt über `similarity_search_with_score()` in `rag_query`. So werden die gefundenen Dokumente nur **einmal** abgerufen und sowohl zur Anzeige als auch als LLM-Kontext verwendet.

In [27]:
# LLM (Kurznotation: "provider:model")
llm = init_chat_model(BASELINE)

# Parser
parser = StrOutputParser()

`format_documents` extrahiert den Textinhalt (`page_content`) aus einer Liste von Dokumentobjekten und verbindet alle Inhalte mit doppelten Zeilenumbrüchen zu einem einzigen String — dem Kontext-Block, der dem LLM übergeben wird.

In [28]:
# Funktion erstelle eine String
def format_documents(documents):
    return "\n\n".join(doc.page_content for doc in documents)

In [29]:
# Answer-Chain ohne Retriever (LCEL)
# Der Kontext wird extern über similarity_search_with_score() bereitgestellt
answer_chain = rag_prompt | llm | parser

print("✅ Answer-Chain erstellt (ohne Retriever)")

✅ Answer-Chain erstellt (ohne Retriever)


<p><font color='black' size="5">
Retrieval-Ergebnisse mit Quelle und Score
</font></p>

In der Praxis ist es wichtig zu wissen, **welche Dokumente** der Retriever gefunden hat und **wie relevant** sie für die Anfrage sind. Die Funktion `rag_query` übernimmt den gesamten RAG-Ablauf in einem optimierten Schritt:

1. **Retrieve mit Score:** `similarity_search_with_score()` liefert die gefundenen Dokumente zusammen mit dem Ähnlichkeits-Score (L2-Distanz bei Chroma – kleinere Werte = höhere Ähnlichkeit)
2. **Kontext formatieren:** Die gefundenen Dokumente werden direkt als Kontext-String aufbereitet
3. **LLM-Antwort:** Die `answer_chain` erhält den fertig formatierten Kontext und generiert die Antwort

**Vorteil gegenüber der Standard-Chain:** Der Retrieve wird nur **einmal** ausgeführt – dieselben Dokumente, die in der Ergebnis-Tabelle angezeigt werden, fließen auch als Kontext in die LLM-Antwort ein. Das spart einen API-Call zum Embedding-Modell und stellt sicher, dass die angezeigte Quelle exakt dem verwendeten Kontext entspricht.

In [30]:
def rag_query(user_input, vectorstore, answer_chain, k=4):
    """RAG-Abfrage mit Anzeige der Retrieval-Ergebnisse (Quelle, Score) und LLM-Antwort.

    Der Retrieve-Schritt erfolgt einmalig über similarity_search_with_score().
    Die gefundenen Dokumente werden sowohl angezeigt als auch direkt als Kontext
    an die answer_chain übergeben – kein doppelter Retrieve.
    """

    # 1. Retrieve: Dokumente mit Score abrufen (einmalig!)
    docs_with_scores = vectorstore.similarity_search_with_score(user_input, k=k)

    # 2. Retrieval-Ergebnisse als Markdown-Tabelle zusammenbauen (ein Block!)
    tabelle = "| Nr. | Score | Quelle | Inhalt (Auszug) |\n"
    tabelle += "|-----|-------|--------|------------------|\n"
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        quelle = doc.metadata.get("source", "unbekannt").split("/")[-1]
        inhalt = doc.page_content[:80].replace("\n", " ").replace("|", "\\|")
        tabelle += f"| {i} | {score:.4f} | {quelle} | {inhalt}... |\n"

    mprint(f"### 🧑 Mensch:\n{user_input}")
    mprint(f"### 🔍 Retrieval-Ergebnisse:\n{tabelle}")

    # 3. Kontext aus den bereits abgerufenen Dokumenten formatieren
    context = format_documents([doc for doc, score in docs_with_scores])

    # 4. LLM-Antwort generieren (mit dem bereits abgerufenen Kontext)
    response = answer_chain.invoke({"context": context, "question": user_input}, config=run_cfg)
    mprint(f"### 🤖 KI:\n{response}")

Abfrage einer Person aus dem Beispieldatensatz:

`rag_query` gibt zunächst die **Retrieval-Ergebnisse** mit Quelle und Score als Tabelle aus, anschließend die **LLM-Antwort** über die RAG-Chain.

In [31]:
rag_query("Was macht Tariq Hassan?", vectorstore, answer_chain)

### 🧑 Mensch:
Was macht Tariq Hassan?

### 🔍 Retrieval-Ergebnisse:
| Nr. | Score | Quelle | Inhalt (Auszug) |
|-----|-------|--------|------------------|
| 1 | 0.7439 | biografien_3.pdf | Tariq Hassan: Tariq Hassan ist ein Sozialer Algorithmus-Therapeut und digitaler ... |
| 2 | 1.2982 | biografien_1.txt | Dorian Blackwood: Dorian Blackwood hat seine Karriere als Zauberer auf den Straß... |
| 3 | 1.2985 | biografien_3.pdf | Rafiq Al-Farsi: Rafiq Al-Farsi ist ein Wüstenökologie-Ingenieur und Wasserharves... |
| 4 | 1.3292 | biografien_3.pdf | Sein Team hat spezielle Programme für Post-Konflikt-Regionen entwickelt, wo sozi... |


### 🤖 KI:
Tariq Hassan ist ein sozialer Algorithmus-Therapeut und digitaler Ethiker, der sich auf die Heilung gesellschaftlicher Polarisierungen spezialisiert hat, die durch manipulative Online-Algorithmen verstärkt werden. Er hat das System „Consensus Bridge“ entwickelt, um toxische Informationsblasen zu identifizieren und Verbindungen zwischen isolierten Gemeinschaften herzustellen. Außerdem gründete er das „Digitale Kohäsions-Institut“, das mit Regierungen und zivilgesellschaftlichen Organisationen zusammenarbeitet, um digitale Spaltungen zu überwinden.

In [32]:
rag_query("Welche Informationen sind zu Elara Fontaine verfügbar?", vectorstore, answer_chain)

### 🧑 Mensch:
Welche Informationen sind zu Elara Fontaine verfügbar?

### 🔍 Retrieval-Ergebnisse:
| Nr. | Score | Quelle | Inhalt (Auszug) |
|-----|-------|--------|------------------|
| 1 | 0.8133 | biografien_1.txt | Elara Fontaine: Elara Fontaine ist eine Pionierin auf dem Gebiet der neurotechno... |
| 2 | 1.2378 | biografien_3.pdf | Weltraumwettervorhersage gefunden, da ihre Sensoren frühe Warnungen vor potenzie... |
| 3 | 1.2564 | biografien_4.docx | Zara El-Amin: Zara El-Amin ist eine Neurosexologin und Intimitätstechnologin, di... |
| 4 | 1.3054 | biografien_3.pdf | Freya Johansson: Freya Johansson ist eine Nordlicht-Energietechnologin und Atmos... |


### 🤖 KI:
Zu **Elara Fontaine** ist verfügbar, dass sie eine Pionierin der neurotechnologischen Traumforschung ist und als **leitende Traumdesignerin bei LucidSphere Labs** arbeitet. Sie entwickelt KI-gestützte Algorithmen, um Menschen das **luzide Träumen** zu erlernen und **gezielt Traum-Erlebnisse** zu steuern, und ihr Team kooperiert mit **Schlafkliniken** zur Hilfe bei Angststörungen und Schlaflosigkeit. Zusätzlich schreibt sie **Kurzgeschichten** aus inspirierten echten Träumen und hält **weltweite Vorträge** über die Zukunft des Träumens.

In [33]:
rag_query("Wer ist Ralf Bendig?", vectorstore, answer_chain)

### 🧑 Mensch:
Wer ist Ralf Bendig?

### 🔍 Retrieval-Ergebnisse:
| Nr. | Score | Quelle | Inhalt (Auszug) |
|-----|-------|--------|------------------|
| 1 | 1.2279 | biografien_2.md | Raoul Mendez: Raoul Mendez ist ein Bioakustiker und Naturschützer, der ein globa... |
| 2 | 1.3334 | biografien_3.pdf | Rafiq Al-Farsi: Rafiq Al-Farsi ist ein Wüstenökologie-Ingenieur und Wasserharves... |
| 3 | 1.3736 | biografien_3.pdf | ein Interesse an der Beziehung zwischen Klang und Lebensqualität. Neben seiner H... |
| 4 | 1.3890 | biografien_3.pdf | Tochter einer Planetengeologin und eines Game-Designers verbindet sie wissenscha... |


### 🤖 KI:
Ich weiß es nicht: In dem gegebenen Kontext kommt „Ralf Bendig“ nicht vor, daher kann ich nicht sagen, wer er ist.

# 6 | RAG-Qualität prüfen
---

<p><font color='black' size="5">
Warum reicht ein Blick auf die Ausgabe nicht?
</font></p>

Eine RAG-Antwort kann flüssig klingen und trotzdem falsch sein — weil der Retriever das falsche Dokument gefunden hat, der Kontext zu knapp war oder das Modell halluziniert. Wer das nicht systematisch prüft, hat keine Aussage darüber, ob eine Änderung am System eine Verbesserung war.

**Die Arbeitsroutine für jede RAG-Entwicklung:**
1. Kleines Eval-Set anlegen (5–10 Fragen mit erwarteten Kernantworten)
2. System durchlaufen lassen und Antworten sammeln
3. Antworten nach klaren Kriterien bewerten (korrekt / teilweise / falsch)
4. Änderung einbauen → erneut messen → vergleichen

**Bewertungskriterien:**

| Kriterium | Frage |
|-----------|-------|
| Inhaltliche Korrektheit | Stimmt die Antwort mit dem erwarteten Kern überein? |
| Quellennutzung | Hat der Retriever das richtige Dokument gefunden? |
| Halluzination | Enthält die Antwort Fakten, die nicht in den Dokumenten stehen? |

Typischer Fehler: Das Eval-Set wird einmal angelegt und dann vergessen. Der Wert liegt im Wiederholen — vor und nach jeder wesentlichen Änderung am System.

In [37]:
# Eval-Set: Fragen aus dem Biografien-Korpus
# Pro Eintrag: Frage + erwarteter Kern der richtigen Antwort
eval_set = [
    {
        "frage": "Was macht Tariq Hassan beruflich?",
        "erwartet": "Sozialer Algorithmus-Therapeut und digitaler Ethiker",
    },
    {
        "frage": "Welche Organisation hat Thoren Navarro gegründet?",
        "erwartet": "Submerged Heritage Project",
    },
    {
        "frage": "Was produziert die Firma UrbanHarvest von Mai Zhang?",
        "erwartet": "Lebensmittel oder Gemüse in vertikalen Farmen",
    },
    {
        "frage": "In welchem Bereich ist Elara Fontaine tätig?",
        "erwartet": "Traumforschung oder Neurotechnologie",
    },
    {
        "frage": "Wer ist Friedrich Merz?",
        "erwartet": "nicht im Korpus vorhanden",  # Negativtest: Person fehlt im Datensatz
    },
]

In [38]:
class EvalBewertung(BaseModel):
    score: float = Field(description="1.0 = korrekt, 0.5 = teilweise korrekt, 0.0 = falsch oder halluziniert")
    begruendung: str = Field(description="Kurze Begründung (ein Satz)")

# Demo: BASELINE; für Produktion: JUDGE (präzisere Urteile)
judge_llm = init_chat_model(BASELINE)

judge_chain = ChatPromptTemplate.from_messages([
    ("system", "Du bewertest RAG-Antworten. Stimmt die Antwort mit dem erwarteten Kern überein? "
               "Score: 1.0 = korrekt, 0.5 = teilweise, 0.0 = falsch oder halluziniert."),
    ("human", "Frage: {frage}\nErwarteter Kern: {erwartet}\nTatsächliche Antwort: {antwort}"),
]) | judge_llm.with_structured_output(EvalBewertung)

# Eval-Schleife
eval_ergebnisse = []

for eintrag in eval_set:
    docs = vectorstore.similarity_search(eintrag["frage"], k=4)
    context = format_documents(docs)
    antwort = answer_chain.invoke({"context": context, "question": eintrag["frage"]}, config=run_cfg)

    bewertung = judge_chain.invoke({
        "frage": eintrag["frage"],
        "erwartet": eintrag["erwartet"],
        "antwort": antwort,
    }, config=run_cfg)

    eval_ergebnisse.append({
        "frage": eintrag["frage"],
        "score": bewertung.score,
        "begruendung": bewertung.begruendung,
    })

In [39]:
zeilen = ["| Frage | Score | Begründung |", "|-------|-------|------------|"]
for e in eval_ergebnisse:
    icon = "✅" if e["score"] == 1.0 else ("⚠️" if e["score"] == 0.5 else "❌")
    frage_kurz = e["frage"][:45] + "…" if len(e["frage"]) > 45 else e["frage"]
    zeilen.append(f"| {frage_kurz} | {icon} {e['score']:.1f} | {e['begruendung']} |")

gesamt = sum(e["score"] for e in eval_ergebnisse) / len(eval_ergebnisse)
mprint("\n".join(zeilen))
mprint(f"\n**Gesamtscore: {gesamt:.0%}** ({len(eval_ergebnisse)} Fragen)")

| Frage | Score | Begründung |
|-------|-------|------------|
| Was macht Tariq Hassan beruflich? | ✅ 1.0 | Die Antwort trifft den erwarteten Kern (Sozialer Algorithmus-Therapeut und digitaler Ethiker) und ergänzt stimmige Details dazu. |
| Welche Organisation hat Thoren Navarro gegrün… | ✅ 1.0 | Die tatsächliche Antwort stimmt mit dem erwarteten Kern überein: Thoren Navarro hat das „Submerged Heritage Project“ gegründet. |
| Was produziert die Firma UrbanHarvest von Mai… | ⚠️ 0.5 | Die Antwort trifft den Kern teilweise (vertikale Farmen und frisches Gemüse), erwähnt aber zusätzlich biolumineszente Pilze und bleibt dabei ohne eindeutige, erwartungskonforme Bestätigung zur Hauptproduktion von UrbanHarvest. |
| In welchem Bereich ist Elara Fontaine tätig? | ✅ 1.0 | Die Antwort stimmt mit dem erwarteten Kern überein und ordnet Elara Fontaine korrekt dem Bereich Traumforschung bzw. neurotechnologischer Traumforschung zu. |
| Wer ist Friedrich Merz? | ✅ 1.0 | Die Antwort stimmt mit dem erwarteten Kern überein: Friedrich Merz ist nicht im Korpus vorhanden und wird daher korrekt nicht beantwortet. |


**Gesamtscore: 90%** (5 Fragen)

**Was bedeutet dieses Ergebnis?**

Ein Gesamtscore ist kein Qualitätszertifikat — er ist ein Ausgangspunkt. Interessant wird er erst im Vergleich: Verbessert sich der Score, wenn die Chunk-Größe angepasst wird? Wenn mehr Dokumente abgerufen werden (`k=6` statt `k=4`)? Wenn der Prompt spezifischer formuliert ist?

Grenze: Ein Eval-Set mit 5 Fragen kann keine statistisch belastbaren Aussagen liefern. Für produktive RAG-Systeme empfehlen sich 20–50 Testfälle und LangSmith Datasets für Versionierung und automatisierten Vergleich.

# 7 | Häufige Probleme und Lösungen
---

| Problem | Symptom | Lösung |
|---------|---------|--------|
| **Niedrige Retrieval-Präzision** | Irrelevante Dokumente gefunden | • Embedding-Modell wechseln<br>• Chunk-Größe anpassen<br>• Hybrid Search verwenden |
| **Halluzinationen** | Fakten nicht in Quellen enthalten | • Temperature reduzieren<br>• Strengere Prompts<br>• Quellentreue in System-Message betonen |
| **Langsame Performance** | Hohe Latenz bei Anfragen | • Vektordatenbank optimieren<br>• Retrieval-Anzahl reduzieren<br>• Caching implementieren |
| **Inkonsistente Antworten** | Verschiedene Antworten auf gleiche Frage | • Temperature auf 0 setzen<br>• Deterministische Retrieval-Reihenfolge<br>• Seed-Parameter verwenden |
| **Context Overflow** | Token-Limit überschritten | • Context Compression<br>• Chunk-Größe reduzieren<br>• Weniger Dokumente abrufen |


# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten sind Anregungen — wer eine andere Herausforderung angeht, ist herzlich eingeladen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll in diesem Kurs auch als Lern- und Entwicklungshilfe genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Ideen für Teilschritte zu entwickeln oder Code-Varianten zu prüfen.
> Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Wende ein einfaches RAG auf einen kleinen eigenen Korpus (mind. 3 Dokumente) an und beantworte eine Frage mit Quellennachweis.

**✅ Erledigt wenn:** Die Antwort enthält eine erkennbare Quellenangabe — nachweisbar aus dem eigenen Korpus, kein allgemeines Modellwissen.

In [ ]:
# Grundlagen: Einfaches RAG auf eigenem Korpus
# Startpunkt: rag_query()-Funktion aus Kapitel 6 als Vorlage

# 1. Mindestens 3 eigene Dokumente laden (txt, pdf oder md)
# 2. Vectorstore erstellen
# 3. Eine Frage stellen, die im Korpus beantwortet werden kann
# 4. Ausgabe: Antwort + Quelle
# ...

**Aufbau**

Beantworte mindestens fünf verschiedene Fragen und dokumentiere die Ergebnisse strukturiert (Frage, Antwort, verwendete Quelle).

**✅ Erledigt wenn:** Alle fünf Antworten enthalten Quelle und Score; bei einer Frage ohne passende Quelle meldet das System das explizit.

In [ ]:
# Aufbau: 5 Fragen beantworten, Ergebnisse dokumentieren
# Startpunkt: rag_query()-Schleife aus Kapitel 6

fragen = [
    "Welches Unternehmen betreibt Unterwasser-Energieanlagen?",
    "Welche Software analysiert spektroskopische Daten zur Atmosphärenforschung?",
    "Welche Technologie hat der namibische Wüstenkäfer inspiriert?",
    "Wie viele Unterwasserstationen betreibt DeepBlue Energy?",
    "Wann wurde der Eiffelturm gebaut?",  # Frage 5: Thema das NICHT im Korpus ist
]

for frage in fragen:
    rag_query(frage, vectorstore, answer_chain)
    print('---')


**Vertiefung**

Lege ein kleines Eval-Set mit erwarteten Antworten an und überprüfe die RAG-Qualität systematisch (mind. 3 Frage-Antwort-Paare mit Bewertung Richtig/Falsch/Teilweise).

**✅ Erledigt wenn:** Das Eval-Set zeigt für alle Testfragen Score und Begründung; der Gesamtscore erscheint am Ende als Prozentzahl.

In [ ]:
# Vertiefung: RAG-Qualität mit Eval-Set messen
# Startpunkt: Eval-Schleife aus Kapitel 7 als Vorlage

eval_set = [
    {"frage": "...", "erwartet": "..."},
    {"frage": "...", "erwartet": "..."},
    {"frage": "...", "erwartet": "..."},  # Negativtest
]

# judge_chain aus Kapitel 7 verwenden
# Gesamtscore berechnen und ausgeben
# ...